Première version : travail sur la qualité de l'air modélisation d'un seuil d'alerte sur la qualité de l'air. 


Module d'alerte complet : 


Bloc : API depuis base de données  fonction : récupérer la moyenne température qualité de l'air et humidité des 10 dernière min via la base de donnée :   appliqué  fiabilsiation concernant l'humidité puis appliquer l'indice atmos enfin création de l'alerte.

In [3]:
import time
from datetime import datetime
import pandas as pd
import smtplib 
import smtplib
from email.message import EmailMessage
import os 
from dotenv import load_dotenv

import os
from dotenv import load_dotenv

# Charge les variables écrites dans le fichier .env
load_dotenv()

USER_EMAIL = os.getenv("USER_EMAIL")
MAIL_PASSWORD = os.getenv("MAIL_PASSWORD")

CHIP_ID = "14990683"
INTERVALLE_SCRAPING = 300  # 300 secondes = 5 minutes


def obtenir_dernieres_mesures(chip_id: str):
    date_du_jour = datetime.now().strftime("%Y-%m-%d")
    url = f"https://api-rrd.madavi.de/data_csv/csv-files/{date_du_jour}/data-esp8266-{chip_id}-{date_du_jour}.csv"

    try:
        # Téléchargement du CSV du jour
        df = pd.read_csv(url, sep=";")
        df["Time"] = pd.to_datetime(df["Time"])

        # Filtrage sur les 20 dernières minutes par rapport à la dernière écriture
        df["Time"] = pd.to_datetime(df["Time"]).dt.tz_localize("UTC").dt.tz_convert("Europe/Paris")
        derniere_date = df["Time"].max()
        seuil = derniere_date - pd.Timedelta(minutes=10)
        df_recent = df[df["Time"] >= seuil]

        # Calcul des moyennes
        pm25  = 80
        pm10 = float(df_recent["SDS_P1"].mean())
        hr = float(df_recent["BME280_humidity"].mean())  # Humidité relative moyenne
        temp =float(df_recent["BME280_temperature"].mean())# Température moyenne
        

        return round(pm25, 2), round(pm10, 2), round(hr, 2), round(temp, 2), derniere_date

    except Exception as e:
        print(f"[ERREUR] Échec de récupération : {e}")
        return None, None, None, None, None 


# Boucle de surveillance semi-temps réel
print("[DÉMARRAGE] Surveillance semi-temps réel via Madavi...")

    

AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

In [ ]:
def envoyer_alerte_email(destinataire, statut, niveau, pm25, pm10):
    # Paramètres de votre boîte mail d'expédition
    expediteur = USER_EMAIL # Votre adresse email
    mot_de_passe = MAIL_PASSWORD  # Mot de passe d'application généré (ex: Google)
    serveur_smtp = "smtp.gmail.com"
    port_smtp = 465  # Port SSL

    msg = EmailMessage()
    msg["Subject"] = f"🚨 Alerte Qualité de l'Air : Niveau {statut} (Score {niveau})"
    msg["From"] = expediteur
    msg["To"] = destinataire

    # Contenu du mail
    contenu = f"""Bonjour,

La qualité de l'air s'est dégradée :
- Statut global : {statut} (Niveau {niveau}/6)
- PM2.5 : {pm25:.2f} µg/m³
- PM10 : {pm10:.2f} µg/m³

Action recommandée : Veuillez aérer ou vérifier la pièce OU activer le purificateur d'air si disponible.
"""
    msg.set_content(contenu)

    try:
        with smtplib.SMTP_SSL(serveur_smtp, port_smtp) as smtp:
            smtp.login(expediteur, mot_de_passe)
            smtp.send_message(msg)
        print(f"[EMAIL ENVOYÉ] Alerte transmise à {destinataire}")
    except Exception as e:
        print(f"[ERREUR EMAIL] Échec de l'envoi : {e}")

In [ ]:
def corriger_humidite_pm25(pm_valeur: float, HR: float, seuil_hr: float = 70.0, c: float = 0.25) -> float:
    """
    Corrige la surestimation des particules (PM2.5 / PM10) due à l'humidité relative.
    
    :param pm_valeur: Mesure brute du capteur (µg/m³)
    :param 
    HR
    : Humidité relative mesurée (%)
    :param seuil_hr: Seuil à partir duquel l'effet de l'eau devient mesurable (défaut: 70%)
    :param c: Facteur empirique hygroscopique (défaut: 0.25)
    :return: Valeur PM corrigée (µg/m³)
    """
    # Si l'humidité est normale, la mesure optique est fiable sans correction
    if HR <= seuil_hr:
        return round(pm_valeur, 2)
    
    # Sécurité pour éviter la division par zéro en cas de saturation extrême (condensation)
    HR_normalisee = min(HR, 98.0) / 100.0
    
    # Calcul du facteur de gonflement hygroscopique
    facteur_grossissement = 1.0 + c * (HR_normalisee ** 2) / (1.0 - HR_normalisee)
    
    pm_corrige = pm_valeur / facteur_grossissement
    return round(pm_corrige, 2)

In [ ]:
def corriger_humidite_pm_PM10(pm_valeur: float, HR: float, seuil_hr: float = 70.0, c: float = 0.18) -> float:
    """
    Corrige la surestimation des particules (PM2.5 / PM10) due à l'humidité relative.
    
    :param pm_valeur: Mesure brute du capteur (µg/m³)
    :param 
    HR
    : Humidité relative mesurée (%)
    :param seuil_hr: Seuil à partir duquel l'effet de l'eau devient mesurable (défaut: 70%)
    :param c: Facteur empirique hygroscopique (défaut: 0.25)
    :return: Valeur PM corrigée (µg/m³)
    """
    # Si l'humidité est normale, la mesure optique est fiable sans correction
    if HR <= seuil_hr:
        return round(pm_valeur, 2)
    
    # Sécurité pour éviter la division par zéro en cas de saturation extrême (condensation)
    HR_normalisee = min(HR, 98.0) / 100.0
    
    # Calcul du facteur de gonflement hygroscopique
    facteur_grossissement = 1.0 + c * (HR_normalisee ** 2) / (1.0 - HR_normalisee)
    
    pm_corrige = pm_valeur / facteur_grossissement
    return round(pm_corrige, 2)

In [ ]:
from datetime import datetime
import time
import pandas as pd

categories = [
    "Bon",
    "Moyen",
    "Dégradé",
    "Mauvais",
    "Très mauvais",
    "Extrêmement mauvais",
]

categories_temperature = [
    "Froid",
    "Bon",
    "Chaud",
    "Insoutenable",
    "Extrêmement chaud",
]
seuils_pm25 = [10, 20, 25, 50, 75]
seuils_pm10 = [20, 40, 50, 100, 150]
seuil_temperature = [0, 10, 20, 30, 40]  # Seuils de température pour la qualification

CHIP_ID = "14990683"
INTERVALLE_SCRAPING = 20  # 5 minutes

print("[DÉMARRAGE] Surveillance semi-temps réel activée...")

user_email = os.getenv("USER_EMAIL")  # Récupération de l'adresse email depuis les variables d'environnement

# Variable de contrôle AVANT la boucle
alerte_deja_envoyee = False

while True:
    # 1. Récupération des données
    pm25, pm10, hr, temp, horodatage = obtenir_dernieres_mesures(CHIP_ID)

    if pm25 is not None and pm10 is not None and hr is not None and temp is not None:

        # 2. Correction hygrométrique si HR > 70%
        if hr > 70:
            print("Humidité relative > 70% : application de la correction de Hänel")
            pm25 = corriger_humidite_pm25(pm25, hr)
            pm10 = corriger_humidite_pm_PM10(pm10, hr)

        # 3. Qualification PM2.5
        pos_pm25 = len(seuils_pm25)
        for i in range(len(seuils_pm25)):
            if pm25 <= seuils_pm25[i]:
                pos_pm25 = i
                break
        niveau_pm25 = pos_pm25 + 1

        # 4. Qualification température
        pos_temperature = len(seuil_temperature)
        for i in range(len(seuil_temperature)):
            if temp <= seuil_temperature[i]:
                pos_temperature = i
                break
        niveau_temperature = pos_temperature + 1

        # 4. Qualification PM10
        pos_pm10 = len(seuils_pm10)
        for i in range(len(seuils_pm10)):
            if pm10 <= seuils_pm10[i]:
                pos_pm10 = i
                break
        niveau_pm10 = pos_pm10 + 1

        # 5. Règle ATMO : on retient le niveau le plus dégradé (le plus élevé)
        niveau_global = max(niveau_pm25, niveau_pm10)
        qualification_globale = categories[niveau_global - 1]

        # --- BLOC D'ALERTE PAR EMAIL ---
    if niveau_global > 1:
        print(f"Alerte : {qualification_globale} (Niveau {niveau_global})")
        
        # Envoi uniquement si on n'a pas déjà notifié
        if not alerte_deja_envoyee:
            envoyer_alerte_email(
                destinataire=user_email,
                statut=qualification_globale,
                niveau=niveau_global,
                pm25=pm25,
                pm10=pm10
            )
            alerte_deja_envoyee = True
    else:
        # Quand l'air redevient "Bon" (Niveau 1), on réarme l'alerte pour le prochain pic
        if alerte_deja_envoyee:
            print("✅ Retour à la normale : alerte réarmée.")
            alerte_deja_envoyee = False

        # 6. Affichage synthétique pour les logs
    print("--------------------------------------------------")
    print(f"[{horodatage}] Mesure sur 20 min (HR: {hr:.1f}%, Temp: {temp:.1f}°C) :")
    print(f" - PM2.5 : {pm25} µg/m³ (Niveau {niveau_pm25} - {categories[pos_pm25]})")
    print(f" - PM10  : {pm10} µg/m³ (Niveau {niveau_pm10} - {categories[pos_pm10]})")
    print(f" - Température : {temp} °C (Niveau {niveau_temperature} - {categories_temperature[pos_temperature]})")
    print(f" => Statut ATMO Global : {qualification_globale} (Niveau {niveau_global})")
    print("--------------------------------------------------")

    # 7. Pause obligatoire indentée DANS la boucle
    time.sleep(INTERVALLE_SCRAPING)

[DÉMARRAGE] Surveillance semi-temps réel activée...


Alerte : Extrêmement mauvais (Niveau 6)
[EMAIL ENVOYÉ] Alerte transmise à Leo.panier55@gmail.com
--------------------------------------------------
[2026-11-08 16:47:28+01:00] Mesure sur 20 min (HR: 47.8%, Temp: 19.7°C) :
 - PM2.5 : 80 µg/m³ (Niveau 6 - Extrêmement mauvais)
 - PM10  : 5.14 µg/m³ (Niveau 1 - Bon)
 - Température : 19.66 °C (Niveau 3 - Chaud)
 => Statut ATMO Global : Extrêmement mauvais (Niveau 6)
--------------------------------------------------
Alerte : Extrêmement mauvais (Niveau 6)
--------------------------------------------------
[2026-11-08 16:49:53+01:00] Mesure sur 20 min (HR: 47.7%, Temp: 19.6°C) :
 - PM2.5 : 80 µg/m³ (Niveau 6 - Extrêmement mauvais)
 - PM10  : 5.23 µg/m³ (Niveau 1 - Bon)
 - Température : 19.63 °C (Niveau 3 - Chaud)
 => Statut ATMO Global : Extrêmement mauvais (Niveau 6)
--------------------------------------------------


KeyboardInterrupt: 

bloc d'alert : 

In [ ]:
# 2. Condition : si le niveau global est supérieur à 1 (c'est-à-dire dégradé / pas "Bon")
if niveau_global > 1:
    print(f"Alerte : {qualification_globale} (Niveau {niveau_global})")

else:
    print(f"Conditions optimales : {qualification_globale} (Niveau {niveau_global}) et Température {categories_temperature[pos_temperature]} (Niveau {niveau_temperature})")



Conditions optimales : Bon (Niveau 1) et Température Chaud (Niveau 3)
